# Boundary-Safe Bronze Text Preparation

This notebook reads one configured gzipped brWaC shard from `data/raw/brwac-clean-1.txt.gz`, splits literal `<END>` boundaries before extraction, preserves source text, derives normalized matching text, and writes bronze segments to Parquet.

In [4]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "src" / "tal_qual").exists():
            return path
    raise RuntimeError("Could not find repository root containing src/tal_qual")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

REPO_ROOT

PosixPath('/home/jovyan/work')

In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("tal-qual-bronze-preparation")
    .getOrCreate()
)

spark.version

'4.1.1'

In [6]:
from tal_qual import BRONZE_OUTPUT_PATH, RAW_CORPUS_INPUT, prepare_bronze_dataframe, write_bronze_parquet

input_path = REPO_ROOT / RAW_CORPUS_INPUT
output_path = REPO_ROOT / BRONZE_OUTPUT_PATH

input_path, output_path

(PosixPath('/home/jovyan/work/data/raw/brwac-clean-1.txt.gz'),
 PosixPath('/home/jovyan/work/data/bronze/brwac_segments'))

In [7]:
from pyspark.sql.functions import col, regexp_extract, substring

bronze_df = prepare_bronze_dataframe(spark, input_path)

bronze_df.printSchema()

preview_df = (
    bronze_df
    .select(
        regexp_extract(col("source_file"), r"([^/]+)$", 1).alias("source_file"),
        "original_line_id",
        "segment_id",
        substring("text_original", 1, 90).alias("text_original_preview"),
        substring("match_text", 1, 90).alias("match_text_preview"),
    )
    .limit(20)
)

display(preview_df.toPandas())

root
 |-- source_file: string (nullable = false)
 |-- original_line_id: long (nullable = false)
 |-- segment_id: integer (nullable = false)
 |-- text_original: string (nullable = false)
 |-- text_normalized: string (nullable = false)
 |-- match_text: string (nullable = false)



,source_file,original_line_id,segment_id,text_original_preview,match_text_preview
0,brwac-clean-1.txt.gz,0,0,Conteúdo recente,conteúdo recente
1,brwac-clean-1.txt.gz,0,1,"ESPUMA MARROM CHAMADA ""NINGUÉM MERECE""","espuma marrom chamada ""ninguém merece"""
2,brwac-clean-1.txt.gz,0,2,"31 de Agosto de 2015, 7:07 , por paulo soavins...","31 de agosto de 2015, 7:07 , por paulo soavins..."
3,brwac-clean-1.txt.gz,0,3,Visualizado 202 vezes,visualizado 202 vezes
4,brwac-clean-1.txt.gz,0,4,JORNAL ELETRÔNICO DA ILHA DO MEL,jornal eletrônico da ilha do mel
5,brwac-clean-1.txt.gz,0,5,Uma espuma marrom escuro tem aparecido com fre...,uma espuma marrom escuro tem aparecido com fre...
6,brwac-clean-1.txt.gz,1,0,"Praça Eugenio Jardim, Rio de Janeiro (Capital)","praça eugenio jardim, rio de janeiro (capital)"
7,brwac-clean-1.txt.gz,1,1,"Praça Eugenio Jardim, Rio de Janeiro, Rio de J...","praça eugenio jardim, rio de janeiro, rio de j..."
8,brwac-clean-1.txt.gz,1,2,"Ótimo apartamento em Copacabana, no corte do C...","ótimo apartamento em copacabana, no corte do c..."
9,brwac-clean-1.txt.gz,1,3,"O apartamento é bem espaçoso, tem 3 quartos e ...","o apartamento é bem espaçoso, tem 3 quartos e ..."


In [8]:
bronze_df.count()

4673057

In [9]:
write_bronze_parquet(bronze_df, output_path)

output_path

PosixPath('/home/jovyan/work/data/bronze/brwac_segments')

In [10]:
written_bronze_df = spark.read.parquet(str(output_path))

written_bronze_df.printSchema()
written_bronze_df.count()

root
 |-- source_file: string (nullable = true)
 |-- original_line_id: long (nullable = true)
 |-- segment_id: integer (nullable = true)
 |-- text_original: string (nullable = true)
 |-- text_normalized: string (nullable = true)
 |-- match_text: string (nullable = true)



4673057

In [11]:
from pyspark.sql.functions import col, regexp_extract, substring

written_preview_df = (
    written_bronze_df
    .select(
        regexp_extract(col("source_file"), r"([^/]+)$", 1).alias("source_file"),
        "original_line_id",
        "segment_id",
        substring("text_original", 1, 90).alias("text_original_preview"),
        substring("match_text", 1, 90).alias("match_text_preview"),
    )
    .limit(20)
)

display(written_preview_df.toPandas())

,source_file,original_line_id,segment_id,text_original_preview,match_text_preview
0,brwac-clean-1.txt.gz,0,0,Conteúdo recente,conteúdo recente
1,brwac-clean-1.txt.gz,0,1,"ESPUMA MARROM CHAMADA ""NINGUÉM MERECE""","espuma marrom chamada ""ninguém merece"""
2,brwac-clean-1.txt.gz,0,2,"31 de Agosto de 2015, 7:07 , por paulo soavins...","31 de agosto de 2015, 7:07 , por paulo soavins..."
3,brwac-clean-1.txt.gz,0,3,Visualizado 202 vezes,visualizado 202 vezes
4,brwac-clean-1.txt.gz,0,4,JORNAL ELETRÔNICO DA ILHA DO MEL,jornal eletrônico da ilha do mel
5,brwac-clean-1.txt.gz,0,5,Uma espuma marrom escuro tem aparecido com fre...,uma espuma marrom escuro tem aparecido com fre...
6,brwac-clean-1.txt.gz,1,0,"Praça Eugenio Jardim, Rio de Janeiro (Capital)","praça eugenio jardim, rio de janeiro (capital)"
7,brwac-clean-1.txt.gz,1,1,"Praça Eugenio Jardim, Rio de Janeiro, Rio de J...","praça eugenio jardim, rio de janeiro, rio de j..."
8,brwac-clean-1.txt.gz,1,2,"Ótimo apartamento em Copacabana, no corte do C...","ótimo apartamento em copacabana, no corte do c..."
9,brwac-clean-1.txt.gz,1,3,"O apartamento é bem espaçoso, tem 3 quartos e ...","o apartamento é bem espaçoso, tem 3 quartos e ..."


In [13]:
random_preview_df = (
    written_bronze_df
    .sample(withReplacement=False, fraction=0.00001, seed=57)
    .select(
        regexp_extract(col("source_file"), r"([^/]+)$", 1).alias("source_file"),
        "original_line_id",
        "segment_id",
        substring("text_original", 1, 90).alias("text_original_preview"),
        substring("match_text", 1, 90).alias("match_text_preview"),
    )
    .limit(20)
)

display(random_preview_df.toPandas())

,source_file,original_line_id,segment_id,text_original_preview,match_text_preview
0,brwac-clean-1.txt.gz,10718,0,Customização: faça um cut off short,customização: faça um cut off short
1,brwac-clean-1.txt.gz,25636,1,"Na década de 1950, meu saudoso pai (JOÃO LEITE...","na década de 1950, meu saudoso pai (joão leite..."
2,brwac-clean-1.txt.gz,30075,12,"e agora Hank tem o apoio de seus lideres, afin...","e agora hank tem o apoio de seus lideres, afin..."
3,brwac-clean-1.txt.gz,36630,0,Como ter orgasmos,como ter orgasmos
4,brwac-clean-1.txt.gz,42190,16,"O presidente da ABCR, Ricardo Pinto Pinheiro, ...","o presidente da abcr, ricardo pinto pinheiro, ..."
5,brwac-clean-1.txt.gz,44053,2,No momento estou ilustrando novamente para pub...,no momento estou ilustrando novamente para pub...
6,brwac-clean-1.txt.gz,45520,67,Sempre achei os argentinos mais inteligentes q...,sempre achei os argentinos mais inteligentes q...
7,brwac-clean-1.txt.gz,57804,1455,ARTIGO 5,artigo 5
8,brwac-clean-1.txt.gz,69794,4,Por que fazer reuniões entre professores?,por que fazer reuniões entre professores?
9,brwac-clean-1.txt.gz,75734,58,Cadastre-se para receber a newsletter e fique ...,cadastre-se para receber a newsletter e fique ...
